# Week 2 · Day 3 — LoRA 模型架构搭建与验证

**模型**：`PrithviMAE`（来自 `../hls-foundation-os/pretrained_models/prithvi_100m/prithvi_mae.py`）  
**权重**：`Prithvi_100M.pt`（与 Week 1 notebook 04 完全相同的加载方式）  
**输入**：`(B, 6, 3, 224, 224)`（T=3，复制单帧）  
**LoRA 位置**：`encoder.blocks.*.attn.qkv`（Week 1 notebook 03 确认）

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Microsoft YaHei')
matplotlib.rcParams['axes.unicode_minus'] = False

import sys
import yaml
import torch
import torch.nn as nn
from pathlib import Path
from peft import LoraConfig, get_peft_model, TaskType

# ── 与 Week 1 完全相同的模型路径 ──────────────────────────────
MODEL_DIR = Path('../hls-foundation-os/pretrained_models/prithvi_100m')
sys.path.insert(0, str(MODEL_DIR))
from prithvi_mae import PrithviMAE   # Week 1 notebook 03/04 使用的类

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {DEVICE}')
print(f'模型目录: {MODEL_DIR.resolve()}')

## 3.1 加载 PrithviMAE（与 Week 1 notebook 04 相同方式）

In [ ]:
# Week 1 notebook 04 的确切加载方式
with open(MODEL_DIR / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# 权重文件：Prithvi_100M.pt，直接就是 state_dict
ckpt = torch.load(MODEL_DIR / 'Prithvi_100M.pt', map_location='cpu')
state_dict = ckpt   # ← 注意：不需要 .get('model', ckpt)，直接就是权重

# 从 state_dict 读取维度（Week 1 notebook 04 的方式）
embed_dim         = state_dict['encoder.norm.weight'].shape[0]      # 768
decoder_embed_dim = state_dict['decoder.decoder_embed.bias'].shape[0]  # 512
num_heads         = embed_dim // 64   # 12

print(f'embed_dim={embed_dim}  decoder_embed_dim={decoder_embed_dim}  num_heads={num_heads}')

prithvi = PrithviMAE(
    img_size=224, patch_size=16, num_frames=3,   # T=3，与训练时相同
    tubelet_size=1, in_chans=6,
    embed_dim=embed_dim, depth=12, num_heads=num_heads,
    decoder_embed_dim=decoder_embed_dim,
    decoder_depth=8, decoder_num_heads=16,
    mlp_ratio=4.0, norm_pix_loss=False,
)
prithvi.load_state_dict(state_dict, strict=False)
prithvi.eval()

total_p = sum(p.numel() for p in prithvi.parameters())
print(f'\nPrithviMAE 加载成功，参数量: {total_p/1e6:.1f}M')

## 3.2 探查 Encoder Attention 层名称

Week 1 notebook 03 已确认：LoRA 插入位置 = `encoder.blocks.*.attn.qkv`

In [ ]:
print('=== encoder.blocks.0 的所有 Linear 层 ===')
for name, module in prithvi.named_modules():
    if 'encoder.blocks.0' in name and isinstance(module, nn.Linear):
        print(f'  {name:<55}  in={module.in_features:<5}  out={module.out_features}')

print('\n=== 验证 LoRA target 路径 ===')
# 检查 qkv 和 proj 是否在正确位置
for name, module in prithvi.named_modules():
    if 'encoder.blocks.0.attn' in name and isinstance(module, nn.Linear):
        print(f'  ✓ {name}  → 将被 LoRA 替换')

## 3.3 分割解码头

**Token 处理**（T=3 时）：
```
输入 (B,6,3,224,224) → forward_encoder → latent (B,589,768)
去 CLS → (B,588,768) → reshape (B,3,196,768)
时间维度取均值 → (B,196,768) → (B,768,14,14)
解码 → (B,1,224,224)
```

In [ ]:
class SegDecoder(nn.Module):
    """(B,768,14,14) → (B,1,224,224) 侵蚀概率图"""
    def __init__(self, embed_dim=768):
        super().__init__()
        self.up1  = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, 2, 2), nn.BatchNorm2d(256), nn.GELU())
        self.up2  = nn.Sequential(
            nn.ConvTranspose2d(256, 64, 4, 4),        nn.BatchNorm2d(64),  nn.GELU())
        self.up3  = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 4),         nn.BatchNorm2d(32),  nn.GELU())
        self.head = nn.Conv2d(32, 1, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        # (B,768,14,14) → (B,1,224,224)
        return self.head(self.up3(self.up2(self.up1(x))))

# 快速形状验证
dec_test = SegDecoder()
out = dec_test(torch.zeros(2, 768, 14, 14))
print(f'Decoder 输出: {out.shape}  ← 期望 (2,1,224,224)')
assert out.shape == (2, 1, 224, 224); print('✓ Decoder 形状正确')

In [ ]:
class PrithviSegModel(nn.Module):
    """
    输入: (B, 6, 224, 224)
    内部: 复制为 (B, 6, 3, 224, 224)，调用 PrithviMAE encoder
    输出: (B, 1, 224, 224) 侵蚀概率 logits
    """
    NUM_FRAMES = 3
    EMBED_DIM  = 768
    GRID_SIZE  = 14   # 224 // 16

    def __init__(self, prithvi_model):
        super().__init__()
        self.prithvi = prithvi_model
        self.decoder = SegDecoder(self.EMBED_DIM)

    def forward(self, x):
        B = x.shape[0]

        # Step 1: 单帧复制为 T=3（与 Week 1 notebook 04 相同）
        x_t = x.unsqueeze(2).repeat(1, 1, self.NUM_FRAMES, 1, 1)  # (B,6,3,224,224)

        # Step 2: Encoder（mask_ratio=0 = 不遮挡）
        latent, _, _ = self.prithvi.forward_encoder(x_t, mask_ratio=0.0)
        # latent: (B, 589, 768)  589 = 1 CLS + 588 patches (196×3)

        # Step 3: 去掉 CLS token
        tokens = latent[:, 1:, :]    # (B, 588, 768)

        # Step 4: reshape → 时间维度取均值
        tokens = tokens.reshape(B, self.NUM_FRAMES,
                                self.GRID_SIZE * self.GRID_SIZE,
                                self.EMBED_DIM)  # (B,3,196,768)
        tokens = tokens.mean(dim=1)              # (B,196,768)

        # Step 5: 转为空间特征图
        feat = tokens.transpose(1, 2).reshape(
            B, self.EMBED_DIM, self.GRID_SIZE, self.GRID_SIZE)  # (B,768,14,14)

        return self.decoder(feat)  # (B,1,224,224)


model = PrithviSegModel(prithvi).to(DEVICE)
print('PrithviSegModel 构建完成')

## 3.4 应用 LoRA

In [ ]:
# target_modules: 匹配 encoder.blocks.*.attn.qkv 和 .proj
# peft 按后缀匹配，'qkv'/'proj' 会命中 encoder.blocks.*.attn.qkv/proj
LORA_R     = 8
LORA_ALPHA = 16

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    bias='none',
    target_modules=['qkv', 'proj'],   # 命中 encoder.blocks.*.attn.qkv/proj
    task_type=TaskType.FEATURE_EXTRACTION,
)

# 只对 prithvi encoder 部分应用 LoRA（MAE decoder 不参与分割训练）
model.prithvi = get_peft_model(model.prithvi, lora_cfg)
model.prithvi.print_trainable_parameters()

# decoder head 默认全部可训练
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n总参数:    {total/1e6:.2f}M')
print(f'可训练:    {trainable/1e6:.2f}M ({100*trainable/total:.1f}%)')

## 3.5 前向传播验证

In [ ]:
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 6, 224, 224).to(DEVICE)
    out   = model(dummy)

print(f'输入: {dummy.shape}')
print(f'输出: {out.shape}  ← 期望 (2,1,224,224)')
print(f'Logits 范围: [{out.min().item():.3f}, {out.max().item():.3f}]')
assert out.shape == (2, 1, 224, 224)
print('\n✓ 前向传播验证通过')

## 3.6 显存测试

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    model.train()
    x = torch.randn(4, 6, 224, 224).to(DEVICE)
    model(x).mean().backward()
    mb = torch.cuda.max_memory_allocated() / 1e6
    print(f'batch=4  峰值显存: {mb:.0f} MB  ({mb/1024:.1f} GB)')
    print(f'RTX 3090 有 24 GB，余量: {24 - mb/1024:.1f} GB')
    torch.cuda.empty_cache()
else:
    print('CPU 模式，跳过显存检查')

print('\n✓ Day 3 完成')

---
## 💾 保存到 GitHub

In [ ]:
import subprocess, os
REPO_DIR = str(Path('..').resolve()); os.chdir(REPO_DIR)
def git(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=REPO_DIR)
    out = (r.stdout + r.stderr).strip()
    if out: print(out)

git('git add notebooks/07_lora_model.ipynb')
git('git commit -m "Week2 Day3: PrithviMAE + LoRA(r=8) on encoder.blocks.attn + SegDecoder verified"')
git('git push origin main')
print('\n✓ 已推送到 GitHub')